# 01 — Does sleep amount / consistency / recent history predict HRV and resting heart rate?

**Question:** ignoring exercise entirely for now, does last night's sleep duration,
this stretch's sleep consistency, and the trailing 7 days of sleep history predict
that morning's HRV (`hrv_rmssd_milli`) and resting heart rate (`resting_heart_rate`)?

Reminder on the pairing (see `src/clean.py`): each row's `resting_heart_rate` and
`hrv_rmssd_milli` come from the sleep at the *end* of that row's cycle — i.e. the
night following that row's `date`, measured on waking. So "last night's sleep"
for a given row's HRV/RHR **is that same row's own sleep columns** — no shifting
needed for the same-night relationship. The 7-day rolling average, however,
deliberately **excludes** the current night, since it's meant to capture
cumulative sleep debt built up *before* that night, not the night itself.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

df = pd.read_csv("../data/daily.csv", parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)
print(f"{len(df)} days, {df['date'].min().date()} to {df['date'].max().date()}")
df.head()

## Build the sleep variables

- `total_sleep_hours` — actual time asleep (light + deep/slow-wave + REM), **not**
  `hours_in_bed` (which also counts time spent awake in bed).
- `sleep_consistency_pct` — already computed by WHOOP, used as-is.
- `prior_1d_avg_sleep_hours` ... `prior_5d_avg_sleep_hours` — rolling mean of
  `total_sleep_hours` over the trailing 1, 2, 3, 4, or 5 nights, **shifted so
  each excludes the current night** (it's "how much sleep debt had built up
  walking into tonight," not tonight itself). Each window is strict - a night
  only gets a value once that many full real prior nights exist
  (`min_periods` == window size, no partial-window averages). Comparing all
  five lags side by side shows *how far back* sleep debt actually matters,
  rather than picking one arbitrary lookback window.

In [ ]:
df["total_sleep_hours"] = df["hours_light_sleep"] + df["hours_deep_sleep"] + df["hours_rem_sleep"]

# shift(1) first so each rolling window is the N nights BEFORE tonight, not including it.
# min_periods == window makes each strict: no value until that many full prior nights exist.
lag_vars = []
for n in range(1, 6):
    col = f"prior_{n}d_avg_sleep_hours"
    df[col] = df["total_sleep_hours"].shift(1).rolling(window=n, min_periods=n).mean()
    lag_vars.append(col)

sleep_vars = ["total_sleep_hours", "sleep_consistency_pct", "sleep_efficiency_pct"] + lag_vars
outcome_vars = ["hrv_rmssd_milli", "resting_heart_rate"]

df[sleep_vars + outcome_vars].describe()

## Correlation matrix

Note: this drops any row missing *any* sleep variable (so n=266, matching the smallest-coverage variable, prior_5d_avg_sleep_hours), while the per-pair table below drops NaNs independently per pair (so some rows show n=270). That is why the r values differ slightly between the two — both are correct, they just use slightly different row sets.

Note: this drops any row missing *any* sleep variable (so n=266, matching the smallest-coverage variable, prior_5d_avg_sleep_hours), while the per-pair table below drops NaNs independently per pair (so some rows show n=270). That is why the r values differ slightly between the two - both are correct, they just use slightly different row sets.

In [ ]:
corr_df = df[sleep_vars + outcome_vars].dropna()
print(f"n = {len(corr_df)} days with complete data")
corr_df.corr(numeric_only=True).loc[sleep_vars, outcome_vars]

## Correlation + significance for each sleep variable vs each outcome

Pearson r alone doesn't tell you whether a relationship is likely real vs. noise —
the p-value does that (rule of thumb: p < 0.05 means "unlikely to be random chance
given this sample size"). Both matter; report both, not just r.

In [ ]:
results = []
for sv in sleep_vars:
    for ov in outcome_vars:
        pair = df[[sv, ov]].dropna()
        r, p = pearsonr(pair[sv], pair[ov])
        results.append({"sleep_var": sv, "outcome": ov, "n": len(pair), "r": round(r, 3), "p_value": round(p, 4)})

results_df = pd.DataFrame(results)
results_df

## What r and p mean, printed on each panel below

Each panel now prints its own `r=...` and `p=...` directly on the plot, so you
can read accuracy off the graph itself instead of cross-referencing the table
above.

- **r (Pearson correlation, -1 to 1)** - how tightly the points hug the red
  trend line, and in which direction. `r=0` means no linear relationship at
  all (a shapeless cloud); `r=-1` or `r=1` means the points fall in a perfect
  line. Sign is direction: negative means "as x goes up, y tends to go down."
  In this data every r is small (|r| < 0.2), which is *why* the clouds below
  look mostly shapeless even where the red line has a visible slope - r is
  telling you the same thing your eyes are: the line is a real but weak lean,
  not a tight relationship.
- **p (p-value, 0 to 1)** - roughly, "if there were truly *no* relationship,
  how likely is it I'd see an r this large just from random day-to-day noise,
  given how many days I have?" Small p (conventionally < 0.05) means unlikely
  to be pure chance - i.e. probably a real, if possibly small, effect. Large p
  means the r you see is well within what random noise alone could produce -
  don't trust that line's slope.
- **Why both matter together**: r tells you *how strong*, p tells you *how
  believable*. A panel can have a decent-looking r from a handful of extreme
  points (unreliable, high p) or a small-but-real r backed by enough days to
  trust it (low p) - the numbers on each panel let you tell those apart at a
  glance rather than eyeballing the cloud.

## Scatter plots: each sleep variable vs. each outcome

In [ ]:
fig, axes = plt.subplots(len(sleep_vars), len(outcome_vars), figsize=(10, 28))

for i, sv in enumerate(sleep_vars):
    for j, ov in enumerate(outcome_vars):
        ax = axes[i, j]
        pair = df[[sv, ov]].dropna()
        ax.scatter(pair[sv], pair[ov], alpha=0.4, s=15)
        if len(pair) > 1:
            m, b = np.polyfit(pair[sv], pair[ov], 1)
            xs = np.linspace(pair[sv].min(), pair[sv].max(), 50)
            ax.plot(xs, m * xs + b, color="red", linewidth=1)
            r, p = pearsonr(pair[sv], pair[ov])
            ax.text(
                0.03, 0.96, f"r={r:.3f}\np={p:.4f}\nn={len(pair)}",
                transform=ax.transAxes, va="top", ha="left", fontsize=8,
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8, edgecolor="gray"),
            )
        ax.set_xlabel(sv)
        ax.set_ylabel(ov)

plt.tight_layout()
plt.savefig("../output/01_sleep_vs_hrv_rhr_scatter.png", dpi=120)
plt.show()

## Why does higher sleep efficiency associate with *lower* HRV? (r=-0.13)

The RHR direction (higher efficiency -> lower RHR, r=-0.12) matches intuition:
a fragmented, low-efficiency night means more awakenings, and each awakening
spikes heart rate, so more fragmentation should mean a higher measured resting
heart rate. That part checks out.

The HRV direction is the confusing one, and here's the most likely reason it
doesn't behave the same way: **`sleep_efficiency_pct` measures a ratio (time
asleep / time in bed), not an absolute amount of recovery.** A night can be
99% efficient while only being 4 hours long (fell asleep instantly, woke
immediately at 4 hours) - highly "efficient," but with far less deep/REM sleep
than a longer, slightly-more-fragmented 8-hour night sitting at 88% efficiency.
HRV recovery is driven largely by *total* deep and REM sleep accumulated, not
by what fraction of time-in-bed was asleep. So a high efficiency % can
actually be flagging a **short** night (efficient use of a short window)
rather than a **good** one, which would drag HRV down despite the efficiency
number looking great.

**Checked directly**: `sleep_efficiency_pct` vs `total_sleep_hours` in this
data comes back r=-0.237, p=0.0001 - a real, significant negative relationship.
Shorter nights genuinely do tend to score as more "efficient" here. That
supports the explanation above: a high efficiency % is partly just flagging a
short night, not necessarily a well-recovered one, which is consistent with
short-but-efficient nights showing lower HRV despite the high efficiency
number.

## Findings

Ran on 270 days (2025-12-19 to 2026-09-14). Actual results from the correlation
table above:

- **The 1-through-5-day lag gradient is the clearest pattern in this notebook.**
  For HRV, the correlation *strengthens monotonically* as the lookback window
  grows: 1-day r=-0.11 (p=0.067, not quite significant) -> 2-day r=-0.10
  (p=0.12) -> 3-day r=-0.12 (p=0.046) -> 4-day r=-0.16 (p=0.010) -> 5-day
  r=-0.19 (p=0.0025, the strongest/most significant result in the whole
  table). Longer sleep-history windows explain more of the HRV variation than
  any single night does - this is a real, structured trend, not noise from
  picking one arbitrary window.
- **Resting heart rate shows essentially nothing at any lag** - every
  prior_Nd_avg_sleep_hours vs resting_heart_rate p-value is above 0.5,
  completely flat across all 5 windows. Whatever the HRV effect above is,
  it does not extend to RHR at all - this is HRV-specific, not a general
  recovery signal.
- **Direction is still counter-intuitive**: all five lags are *negative* -
  more trailing sleep associates with *lower* HRV, not higher. Combined with
  the monotonic strengthening, this looks less like noise and more like a
  real but backwards-from-hypothesis relationship - worth investigating
  *why* rather than dismissing. Leading candidate: reverse causality (heavier
  training blocks could drive both more recovery sleep *and* lower HRV,
  rather than sleep amount itself lowering HRV).
- **sleep_efficiency_pct is the only same-night variable significant against
  both outcomes** (HRV: r=-0.13, p=0.033; RHR: r=-0.12, p=0.044), also
  counter-intuitive in direction (higher efficiency -> lower HRV/RHR here).
- **total_sleep_hours (last night specifically) and sleep_consistency_pct
  show no significant relationship with either outcome** - the two variables
  that seemed most obviously important going in turned out to have no signal
  on their own; it's the accumulated multi-day history that matters, not any
  single night.
- Simple takeaway: single-night sleep amount is not the driver of next-day
  HRV/RHR in this data, but accumulated sleep history over ~5 days is a real,
  HRV-specific signal worth a dedicated follow-up (e.g. why is it negative,
  and does it survive controlling for training load?). Next natural notebook:
  bring exercise load back in as a control (this one explicitly excluded it).
